### Step 1: Upload and Inspect the Heart Disease Dataset


In [11]:
import io
import pandas as pd
from google.colab import files

print("Upload cardio_train.csv")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No CSV file uploaded.")

filename = next(iter(uploaded))
content = uploaded[filename]

# cardio_train.csv uses semicolon delimiter.
df = pd.read_csv(io.BytesIO(content), sep=";")

required_columns = {
    "age", "gender", "height", "weight",
    "ap_hi", "ap_lo", "cholesterol", "gluc",
    "smoke", "alco", "active", "cardio"
}

missing = required_columns - set(df.columns)
if missing:
    raise ValueError(
        f"Missing columns: {sorted(missing)}. "
        "Check that the correct cardiovascular dataset was uploaded."
    )

print("Loaded:", filename)
print("Shape:", df.shape)
display(df.head())
print(df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("Duplicate rows:", df.drop(
    columns=["id"], errors="ignore"
).duplicated().sum())
print("\nTarget distribution:\n", df["cardio"].value_counts())

Upload cardio_train.csv


Saving cardio_train.csv to cardio_train.csv
Loaded: cardio_train.csv
Shape: (70000, 13)


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


id               int64
age              int64
gender           int64
height           int64
weight         float64
ap_hi            int64
ap_lo            int64
cholesterol      int64
gluc             int64
smoke            int64
alco             int64
active           int64
cardio           int64
dtype: object

Missing values:
 id             0
age            0
gender         0
height         0
weight         0
ap_hi          0
ap_lo          0
cholesterol    0
gluc           0
smoke          0
alco           0
active         0
cardio         0
dtype: int64
Duplicate rows: 24

Target distribution:
 cardio
0    35021
1    34979
Name: count, dtype: int64


### Step 2: Data Cleaning and Feature Engineering


In [12]:
# Work on a copy to keep the original dataset unchanged.
df = df.copy()

# cardio_train.csv age is recorded in days.
df["age"] = df["age"] / 365.25

# Remove duplicates, ignoring database ID.
before = len(df)
df = df.drop_duplicates(
    subset=df.columns.difference(["id"])
)
print("Duplicate rows removed:", before - len(df))

# Apply explicit blood-pressure data-quality filters.
before = len(df)
df = df[
    df["ap_hi"].between(80, 220)
    & df["ap_lo"].between(40, 120)
    & (df["ap_hi"] > df["ap_lo"])
].copy()
print("Rows removed by BP filters:", before - len(df))

# Check remaining data before training.
print("Cleaned shape:", df.shape)
print("Missing values:", int(df.isnull().sum().sum()))
print("Target distribution:\n", df["cardio"].value_counts())
display(df.describe())

Duplicate rows removed: 24
Rows removed by BP filters: 1383
Cleaned shape: (68593, 13)
Missing values: 0
Target distribution:
 cardio
0    34671
1    33922
Name: count, dtype: int64


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
count,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000,68593.000000
mean,49966.824472,53.290492,1.348592,164.360605,74.112409,126.627426,81.271383,1.364585,1.225650,0.087910,0.053314,0.803304,0.494540
std,28846.602781,6.757866,0.476528,8.184829,14.322769,16.564640,9.332790,0.678857,0.571556,0.283166,0.224661,0.397504,0.499974
min,0.000000,29.563313,1.000000,55.000000,11.000000,80.000000,40.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,24993.000000,48.342231,1.000000,159.000000,65.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
50%,49999.000000,53.938398,1.000000,165.000000,72.000000,120.000000,80.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000
75%,74860.000000,58.384668,2.000000,170.000000,82.000000,140.000000,90.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000
max,99999.000000,64.922656,2.000000,250.000000,200.000000,220.000000,120.000000,3.000000,3.000000,1.000000,1.000000,1.000000,1.000000


### Step 3: Train-Test Split and target leak avoidance


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

target_column = "cardio"

numeric_features = [
    "age", "height", "weight", "ap_hi", "ap_lo"
]
categorical_features = [
    "gender", "cholesterol", "gluc",
    "smoke", "alco", "active"
]
feature_names = numeric_features + categorical_features

X = df[feature_names].copy()
y = df[target_column].astype(int)

if X.isnull().any().any():
    raise ValueError("Missing feature values must be handled before training.")

if not set(y.unique()).issubset({0, 1}):
    raise ValueError("Target must contain only binary labels 0 and 1.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            drop="first"
        ), categorical_features)
    ]
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Feature order:", feature_names)

Train: (54874, 11) Test: (13719, 11)
Feature order: ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']


### Step 4: Build Preprocessing Pipeline
We use a `ColumnTransformer` to scale numerical features and handle categorical encodings, wrapped in a reusable scikit-learn Pipeline.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Identify numerical and categorical features
numeric_features = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']
categorical_features = ['gender', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

# Numeric Transformer: Standardize scales
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Categorical Transformer: One-hot encode multiclass variables if any
# Since gender/cholesterol/gluc are already numeric codes, we'll ensure they are encoded correctly
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Combine into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

### Step 5: Model Training, Selection, and Evaluation
We will compare a Random Forest Classifier and a Logistic Regression Model using Cross-Validation on the training set, then evaluate the final champion model on the test set.

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        max_depth=10,
        n_jobs=-1
    )
}

cv_scores = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    scores = cross_val_score(
        pipeline,
        X_train, y_train,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_scores[name] = scores.mean()
    print(
        f"{name}: mean CV ROC-AUC={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )

best_name = max(cv_scores, key=cv_scores.get)
champion_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", models[best_name])
])

champion_pipeline.fit(X_train, y_train)

y_pred = champion_pipeline.predict(X_test)
y_proba = champion_pipeline.predict_proba(X_test)[:, 1]

test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0))
}

print("\nSelected model:", best_name)
print("Test metrics:", test_metrics)
print("\nClassification report:\n",
      classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Logistic Regression: mean CV ROC-AUC=0.7912, std=0.0028
Random Forest: mean CV ROC-AUC=0.8000, std=0.0039

Selected model: Random Forest
Test metrics: {'accuracy': 0.7325606822654712, 'roc_auc': 0.8000442534400036, 'precision': 0.7598398932621748, 'recall': 0.6714812085482682, 'f1': 0.7129332603082701}

Classification report:
               precision    recall  f1-score   support

           0       0.71      0.79      0.75      6934
           1       0.76      0.67      0.71      6785

    accuracy                           0.73     13719
   macro avg       0.74      0.73      0.73     13719
weighted avg       0.74      0.73      0.73     13719

Confusion matrix:
 [[5494 1440]
 [2229 4556]]


### Step 6: Save the Pipeline and Generate the Config Report
We save the fully fitted pipeline to `heart_disease_pipeline.pkl` and create a `metadata.json` describing the features for the VS Code project.

In [16]:
import json
import joblib
import sklearn
import pandas as pd
from pathlib import Path

output_dir = Path("heart_disease")
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "heart_disease_pipeline.pkl"
metadata_path = output_dir / "heart_disease_metadata.json"

# Save the entire fitted preprocessing + model pipeline.
joblib.dump(champion_pipeline, model_path)

metadata = {
    "model_name": best_name,
    "disease_name": "Cardiovascular Disease",
    "dataset_target": "cardio",
    "target_mapping": {
        "0": "No cardiovascular disease label",
        "1": "Cardiovascular disease label"
    },
    "feature_names": feature_names,
    "numerical_features": numeric_features,
    "categorical_features": categorical_features,
    "age_unit": "years",
    "selection_metric": "5-fold cross-validation ROC-AUC",
    "cv_roc_auc": cv_scores,
    "test_metrics": test_metrics,
    "threshold": 0.5,
    "sklearn_version": sklearn.__version__,
    "pandas_version": pd.__version__,
    "input_example": {
        "age": 50.0,
        "gender": 1,
        "height": 168,
        "weight": 75.0,
        "ap_hi": 120,
        "ap_lo": 80,
        "cholesterol": 1,
        "gluc": 1,
        "smoke": 0,
        "alco": 0,
        "active": 1
    }
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", model_path)
print("Saved:", metadata_path)

Saved: heart_disease/heart_disease_pipeline.pkl
Saved: heart_disease/heart_disease_metadata.json


### Step 7: Local Inference Test
Load the saved pipeline file and demonstrate how to use it with example data.

In [17]:
import joblib
import pandas as pd

loaded_pipeline = joblib.load(
    "heart_disease/heart_disease_pipeline.pkl"
)

sample = {
    "age": 50.4,
    "gender": 1,
    "height": 165,
    "weight": 70.0,
    "ap_hi": 130,
    "ap_lo": 85,
    "cholesterol": 2,
    "gluc": 1,
    "smoke": 0,
    "alco": 0,
    "active": 1
}

# Enforce training feature order.
expected_features = metadata["feature_names"]
sample_df = pd.DataFrame([sample])[expected_features]

prediction = int(loaded_pipeline.predict(sample_df)[0])
probability = float(
    loaded_pipeline.predict_proba(sample_df)[0][1]
)

print("Prediction:", prediction)
print("Probability of class 1:", round(probability, 4))
print("Class meaning:", metadata["target_mapping"][str(prediction)])

Prediction: 1
Probability of class 1: 0.5481
Class meaning: Cardiovascular disease label


### Step 8: Download Model & Metadata for VS Code
Use this block to download the artifacts and copy them to your Windows project structure.

In [18]:
from google.colab import files

print("Downloading model and metadata files to your machine...")


files.download("heart_disease/heart_disease_pipeline.pkl")
files.download("heart_disease/heart_disease_metadata.json")

print("\n--- INSTRUCTIONS FOR WINDOWS VS CODE ---")
print("1. Create the target directory inside your VS Code project folder:")
print("   models\\heart_disease\\")
print("2. Move the downloaded 'heart_disease_pipeline.pkl' and 'heart_disease_metadata.json' to that directory.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- INSTRUCTIONS FOR WINDOWS VS CODE ---
1. Create the target directory inside your VS Code project folder:
   models\heart_disease\
2. Move the downloaded 'heart_disease_pipeline.pkl' and 'heart_disease_metadata.json' to that directory.
